In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_curve, roc_auc_score, log_loss, accuracy_score,
    PrecisionRecallDisplay, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import label_binarize

from src.common.load_data import load_data, drop_targets, get_test_val_sets
from src.common.treat_missing import TabularPreprocessor, PreprocessConfig
from src.common.evaluations import evaluate_classification
from src.utils.plots import plot_loss_curve

import warnings

**Understanding the Approved_Flag Tiers (P1 to P4)**

The dataset segments the risk ranking from lowest default risk to highest default risk into four classes:

🟢 **P1 (Premium / Best Customers)**: These are excellent, lowest-risk customers. They possess top-tier CIBIL credit footprints, exceptional repayment histories, and are automatically fast-tracked for unconditional approval.

🟡 **P2 (Good Customers)**: These are highly reliable, low-risk customers. They satisfy the core banking lending criteria safely and are standard "Approved" applicants. (Note: In this specific Kaggle dataset, P2 typically represents the largest share/majority class of accepted loans).

🟠 **P3 (Marginal / Conditional Customers)**: These are medium-risk customers. They generally trigger a "Conditional Approval" or require manual underwriting adjustments (e.g., lower loan amount, higher interest rate, or extra collateral).

🔴 **P4 (Bad / High-Risk Customers)**: These are high-risk customers. They have poor credit histories or high indicators of delinquency and are categorized under "Refer" or "Reject" status.

Unlike the binary problem (approved vs not), this is a genuine **4-class, imbalanced** problem. P2 tends to dominate the dataset, while **P3 is typically squeezed between P1 and P4** in size — this is the class we pay closest attention to below when discussing `class_weight`.

In [ ]:
pd.options.display.max_rows = 200

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
df = load_data("./data/cibil_score/cibil_score.csv")
X, y_lin_reg, y_binary, y_multiclass = drop_targets(df)
X_train, X_val, X_test, y_train, y_val, y_test = get_test_val_sets(X, y_multiclass, test_size=0.3)

## Replace Missing Values

In [ ]:
high_missing = [
    "cc_utilization",
    "pl_utilization"
]
median_columns = [
    "age_oldest_tl",
    "age_newest_tl",
    "pct_currentbal_all_tl",
    "time_since_recent_payment"
]
"""
Delinquency:
Features like num_times_30p_dpd or delinq_2yrs count how many 
separate instances a person crossed that late-payment threshold 
over a specific timeframe (such as the past 6 months, 12 months, or 2 years).
"""
delinquency_columns = [
    "max_delinquency_level",
    "max_deliq_6mts",
    "max_deliq_12mts",
    "time_since_recent_deliquency",
    "time_since_first_deliquency"
]

# If there is no entry means no enquiry
enquiry_columns = [
    "tot_enq",
    "cc_enq",
    "pl_enq",
    "cc_enq_l6m",
    "cc_enq_l12m",
    "pl_enq_l6m",
    "pl_enq_l12m",
    "enq_l3m",
    "enq_l6m",
    "enq_l12m"
]

# Maximum unsecured exposure = 0%
# Customer with no unsecured loan
max_unsecur = ["max_unsec_exposure_inpct"]

zero_cols = max_unsecur + enquiry_columns + delinquency_columns

log_transform_cols = ["netmonthlyincome"]

In [ ]:
config = PreprocessConfig(
        high_missing=high_missing,
        median_cols=median_columns,
        zero_cols=zero_cols,
        log_transform_cols=log_transform_cols,
    )

In [ ]:
preprocessor = TabularPreprocessor(config)
X_train_processed, X_val_processed, X_test_processed = preprocessor.fit_transform(
    X_train, X_val, X_test
)

## Check Class Imbalance (P1 / P2 / P3 / P4)

Before fitting anything, look at how skewed the four classes are. P2 is expected to be the majority class; P3 is the class squeezed between P1 (best) and P4 (worst), and typically has far fewer examples than either neighbour. This is exactly the kind of imbalance `class_weight` is meant to help with, so we check it first and refer back to it throughout.

In [ ]:
class_counts = y_train.value_counts().sort_index()
class_props = (class_counts / class_counts.sum()).round(3)

print("Training class distribution (counts):")
print(class_counts)
print("\nTraining class distribution (proportion):")
print(class_props)

plt.figure(figsize=(6, 4))
class_counts.sort_index().plot(kind="bar", color=["#2ca02c", "#f4d35e", "#ff7f0e", "#d62728"])
plt.title("Class Distribution - Approved Flag (Train)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

## Baseline Multiclass Logistic Regression (`class_weight=None`)

`LogisticRegression` handles more than two classes natively — with `solver="lbfgs"` (and the other solvers besides `liblinear`) it fits a multinomial (softmax) model rather than training separate one-vs-rest binary models. We start with no class weighting as the baseline to compare against later.

In [ ]:
model = LogisticRegression(max_iter=500, solver="lbfgs", random_state=42)
model.fit(X_train_processed, y_train)
y_pred = model.predict(X_val_processed)
y_prob = model.predict_proba(X_val_processed)

In [ ]:
evaluate_classification(model, X_val_processed, y_val, binary=False)

In [ ]:
print(classification_report(y_val, y_pred))

cm = confusion_matrix(y_val, y_pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix - Baseline (class_weight=None)")
plt.show()

## Multiclass ROC-AUC (One-vs-Rest)

ROC/AUC is defined for binary classifiers, so for 4 classes we binarize each class against the rest ("one-vs-rest"), plot one ROC curve per class, and summarise overall performance with the macro-averaged AUC.

In [ ]:
classes = model.classes_
y_val_bin = label_binarize(y_val, classes=classes)

auc_ovr_macro = roc_auc_score(y_val, y_prob, multi_class="ovr", average="macro")
print("Macro-average One-vs-Rest AUC =", auc_ovr_macro)

plt.figure(figsize=(7, 6))
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_val_bin[:, i], y_prob[:, i])
    auc_cls = roc_auc_score(y_val_bin[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, linewidth=2, label=f"{cls} vs rest (AUC = {auc_cls:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="red", label="Random Classifier")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("One-vs-Rest ROC Curves by Class")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

## Train vs Validation Loss / Accuracy Curves (Baseline, `class_weight=None`)

In [ ]:
train_loss = []
val_loss = []
train_accuracy = []
val_accuracy = []

for i in range(1, 30):

    model = LogisticRegression(
        max_iter=i,
        solver="lbfgs",
        random_state=42
    )

    model.fit(X_train_processed, y_train)

    train_prob = model.predict_proba(X_train_processed)
    val_prob = model.predict_proba(X_val_processed)
    train_pred = model.predict(X_train_processed)
    val_pred = model.predict(X_val_processed)

    train_loss.append(
        log_loss(y_train, train_prob, labels=model.classes_)
    )

    val_loss.append(
        log_loss(y_val, val_prob, labels=model.classes_)
    )

    train_accuracy.append(
        accuracy_score(y_train, train_pred)
    )

    val_accuracy.append(
        accuracy_score(y_val, val_pred)
    )


plot_loss_curve(
    train_loss,
    val_loss,
    "Multiclass Logistic Regression - Log Loss"
)

In [ ]:
plot_loss_curve(
    train_accuracy,
    val_accuracy,
    "Multiclass Logistic Regression - Train vs Val Accuracy"
)

## Logistic Regression Parameters

**C** : Inverse of regularization strength; must be a positive float. Like in support vector machines, smaller values specify stronger regularization. *default=1.0*

**C=np.inf results in unpenalized logistic regression.**

**penalty : {'l1', 'l2', 'elasticnet', None}, default='l2'**
Specify the norm of the penalty:

None: no penalty is added;
'l2': add an L2 penalty term and it is the default choice;
'l1': add an L1 penalty term;
'elasticnet': both L1 and L2 penalty terms are added.

**l1_ratio : float, default=0.0**
The Elastic-Net mixing parameter, with 0 <= l1_ratio <= 1. Setting l1_ratio=1 gives a pure L1-penalty, setting l1_ratio=0 a pure L2-penalty. Any value between 0 and 1 gives an Elastic-Net penalty of the form l1_ratio * L1 + (1 - l1_ratio) * L2

**class_weight : dict or 'balanced', default=None**
Weights associated with classes in the form {class_label: weight}. If not given, all classes are supposed to have weight one.

The "balanced" mode uses the values of y to automatically adjust weights inversely proportional to class frequencies in the input data as n_samples / (n_classes * np.bincount(y)).

Note that these weights will be multiplied with sample_weight (passed through the fit method) if sample_weight is specified.

**In a 4-class problem this matters more than in the binary case**: with `class_weight=None`, the loss is dominated by whichever class has the most rows (typically P2), so the optimiser has little incentive to get P3 right. `class_weight="balanced"` reweights each class's contribution to the loss inversely to its frequency, which pushes P3 (and any other minority class) to matter as much as P1/P2/P4 during training.

The choice of the algorithm depends on the penalty chosen (l1_ratio=0 for L2-penalty, l1_ratio=1 for L1-penalty and 0 < l1_ratio < 1 for Elastic-Net) and on (multinomial) multiclass support:

**solver : {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'**
Algorithm to use in the optimization problem. Default is 'lbfgs'.

**Solver and l1_ration support:**

solver	l1_ratio	multinomial multiclass

'lbfgs'	l1_ratio=0	yes

'liblinear'	l1_ratio=1 or l1_ratio=0	no

'newton-cg'	l1_ratio=0	yes

'newton-cholesky'	l1_ratio=0	yes

'sag'	l1_ratio=0	yes

'saga'	0<=l1_ratio<=1	yes


## Regularization — With and Without Class Weights

Because P3 sits between P1 and P4 and is under-represented relative to them (and dwarfed by P2), each regularization variant below is fit **twice**: once with `class_weight=None` (baseline) and once with `class_weight="balanced"`. We compare per-class precision/recall/F1 — especially for **P3** — rather than relying on overall accuracy, since accuracy is dominated by the majority class and can look fine even while P3 is being predicted poorly.

### Ridge regression (L2)

penalty = 'l2'

solver = 'saga'

C = 100.0

In [ ]:
ridge_model = LogisticRegression(
    C=100.0, penalty="l2", max_iter=500, solver="saga", random_state=42
)
ridge_model.fit(X_train_processed, y_train)

print("Ridge (L2) - class_weight=None")
evaluate_classification(ridge_model, X_val_processed, y_val, binary=False)
print(classification_report(y_val, ridge_model.predict(X_val_processed)))

In [ ]:
ridge_model_balanced = LogisticRegression(
    C=100.0, penalty="l2", max_iter=500, solver="saga",
    class_weight="balanced", random_state=42
)
ridge_model_balanced.fit(X_train_processed, y_train)

print("Ridge (L2) - class_weight='balanced'")
evaluate_classification(ridge_model_balanced, X_val_processed, y_val, binary=False)
print(classification_report(y_val, ridge_model_balanced.predict(X_val_processed)))

### Lasso regression (L1)

penalty = 'l1'

solver = 'saga' (only solver that supports 'l1')

C = 1.0

In [ ]:
lasso_model = LogisticRegression(
    C=1.0, penalty="l1", max_iter=500, solver="saga", random_state=42
)
lasso_model.fit(X_train_processed, y_train)

print("Lasso (L1) - class_weight=None")
evaluate_classification(lasso_model, X_val_processed, y_val, binary=False)
print(classification_report(y_val, lasso_model.predict(X_val_processed)))

In [ ]:
lasso_model_balanced = LogisticRegression(
    C=1.0, penalty="l1", max_iter=500, solver="saga",
    class_weight="balanced", random_state=42
)
lasso_model_balanced.fit(X_train_processed, y_train)

print("Lasso (L1) - class_weight='balanced'")
evaluate_classification(lasso_model_balanced, X_val_processed, y_val, binary=False)
print(classification_report(y_val, lasso_model_balanced.predict(X_val_processed)))

### Elasticnet regression

penalty = 'elasticnet'

solver = 'saga' (only solver that supports 'elasticnet')

l1_ratio = 0.7, C = 1.0

In [ ]:
elastic_model = LogisticRegression(
    C=1.0, penalty="elasticnet", l1_ratio=0.7, max_iter=500,
    solver="saga", random_state=42
)
elastic_model.fit(X_train_processed, y_train)

print("Elasticnet - class_weight=None")
evaluate_classification(elastic_model, X_val_processed, y_val, binary=False)
print(classification_report(y_val, elastic_model.predict(X_val_processed)))

In [ ]:
elastic_model_balanced = LogisticRegression(
    C=1.0, penalty="elasticnet", l1_ratio=0.7, max_iter=500,
    solver="saga", class_weight="balanced", random_state=42
)
elastic_model_balanced.fit(X_train_processed, y_train)

print("Elasticnet - class_weight='balanced'")
evaluate_classification(elastic_model_balanced, X_val_processed, y_val, binary=False)
print(classification_report(y_val, elastic_model_balanced.predict(X_val_processed)))

## Grid Search for Best Parameters

As with the binary notebook, we search each penalty family separately (since only certain solvers support certain penalties), but here `class_weight` is included as a search dimension so the grid can pick whichever weighting scheme actually helps. We score on **`f1_macro`** rather than accuracy or `roc_auc`, since macro-F1 treats every class — including the minority P3 — equally, instead of letting the majority class dominate the score.

In [ ]:
# Different penalties only work with certain solvers (see the parameter
# notes above), so we search each penalty family separately instead of
# one grid with invalid solver/penalty combinations. class_weight is
# searched in every branch so the grid can tell us whether balancing
# actually helps once C and the penalty are also tuned.
param_grid = [
    {"penalty": ["l2"], "solver": ["lbfgs"],
     "C": [0.01, 0.1, 1, 10, 100], "max_iter": [200],
     "class_weight": [None, "balanced"]},
    {"penalty": ["l1"], "solver": ["saga"],
     "C": [0.01, 0.1, 1, 10, 100], "max_iter": [200],
     "class_weight": [None, "balanced"]},
    {"penalty": ["elasticnet"], "solver": ["saga"],
     "C": [0.01, 0.1, 1, 10, 100], "l1_ratio": [0.3, 0.5, 0.7],
     "max_iter": [200], "class_weight": [None, "balanced"]},
]

grid_search = GridSearchCV(
    estimator=LogisticRegression(random_state=42),
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    refit=True,
    verbose=1,
    error_score="raise",
)

try:
    grid_search.fit(X_train_processed, y_train)
except Exception as exc:
    raise RuntimeError(f"GridSearchCV failed: {exc}") from exc

print("Best parameters:", grid_search.best_params_)
print(f"Best CV F1-macro: {grid_search.best_score_:.4f}")

cv_results = pd.DataFrame(grid_search.cv_results_).sort_values("rank_test_score")
cv_results[["params", "mean_test_score", "std_test_score", "rank_test_score"]].head(10)

## Run Logistic Regression with Best Parameters
- Refit with the best parameters found (including whichever `class_weight` the grid preferred)
- Run full evaluation, with special attention to P3
- Plot train-validation loss, precision-recall, ROC-AUC, train-val accuracy

In [ ]:
train_loss = []
val_loss = []
train_accuracy = []
val_accuracy = []

best_params = grid_search.best_params_
print("Using best parameters from grid search:", best_params)

for i in range(1, 30):
    params = dict(best_params)
    params["max_iter"] = i
    params["random_state"] = 42
    model = LogisticRegression(**params)
    model.fit(X_train_processed, y_train)

    train_prob = model.predict_proba(X_train_processed)
    val_prob = model.predict_proba(X_val_processed)
    train_pred = model.predict(X_train_processed)
    val_pred = model.predict(X_val_processed)

    train_loss.append(
        log_loss(y_train, train_prob, labels=model.classes_)
    )

    val_loss.append(
        log_loss(y_val, val_prob, labels=model.classes_)
    )

    train_accuracy.append(
        accuracy_score(y_train, train_pred)
    )

    val_accuracy.append(
        accuracy_score(y_val, val_pred)
    )


plot_loss_curve(
    train_loss,
    val_loss,
    "Multiclass Logistic Regression - Log Loss (Best Params)"
)

metrics = evaluate_classification(model, X_val_processed, y_val, binary=False)
print(metrics)
print(classification_report(y_val, model.predict(X_val_processed)))

train_loss_best = log_loss(y_train, model.predict_proba(X_train_processed), labels=model.classes_)
val_loss_best = log_loss(y_val, model.predict_proba(X_val_processed), labels=model.classes_)
print(f"Grid-search best model | Train loss: {train_loss_best:.4f} | Val loss: {val_loss_best:.4f}")

## Precision-Recall: Baseline vs Balanced Best Model (per class)

Since P3 is the minority class squeezed between P1 and P4, its precision-recall curve (one-vs-rest) is the most telling comparison between the unweighted baseline and the grid-search best (which may use `class_weight="balanced"`).

In [ ]:
classes = model.classes_
y_val_bin = label_binarize(y_val, classes=classes)
y_score_best = model.predict_proba(X_val_processed)

fig, axes = plt.subplots(1, len(classes), figsize=(5 * len(classes), 5), sharey=True)
for ax, cls_idx, cls in zip(axes, range(len(classes)), classes):
    PrecisionRecallDisplay.from_predictions(
        y_val_bin[:, cls_idx], y_score_best[:, cls_idx], ax=ax, name="Best model"
    )
    ax.set_title(f"Class {cls} (one-vs-rest)")
plt.tight_layout()
plt.show()

## todo: Your understanding on above Precision-Recall Curves, and how P3 specifically behaves compared to P1/P4

## Get Columns Whose Gradient Becomes 0 from Lasso Regression

In [ ]:
feature_names = np.array(preprocessor.get_feature_names())

if feature_names.shape[0] != X_train_processed.shape[1]:
    raise ValueError(
        f"Feature name count ({feature_names.shape[0]}) does not match "
        f"the number of processed columns ({X_train_processed.shape[1]}); "
        f"the fitted preprocessor and X_train_processed are out of sync."
    )

if "lasso_model_balanced" not in dir():
    raise NameError(
        "lasso_model_balanced is not defined. Run the Lasso regression cell above first."
    )

# For multinomial logistic regression, coef_ has shape (n_classes, n_features):
# one row of weights per class. A feature only truly drops out of the model
# if Lasso zeroed its weight for every class at once.
lasso_coefs = lasso_model_balanced.coef_

zero_weight_mask = np.all(np.isclose(lasso_coefs, 0.0), axis=0)
kept_features_mask = ~zero_weight_mask

zero_weight_features = feature_names[zero_weight_mask]
kept_feature_names = feature_names[kept_features_mask]

print(
    f"Lasso (L1, class_weight='balanced') shrank {zero_weight_mask.sum()} of "
    f"{len(feature_names)} features to exactly 0 across ALL classes:\n"
)
if zero_weight_mask.sum() == 0:
    print("  (none -- every feature kept a non-zero weight for at least one class)")
else:
    for name in zero_weight_features:
        print(f"  - {name}")

print(f"\n{kept_features_mask.sum()} features remain with a non-zero weight for at least one class.")

## Run Logistic Regression Excluding Features Dropped by Lasso

In [ ]:
if kept_features_mask.sum() == 0:
    raise ValueError(
        "All features were shrunk to 0 by Lasso; cannot fit a model with zero features. "
        "Try a smaller C (weaker regularisation) in the Lasso cell above."
    )

X_train_reduced = X_train_processed[:, kept_features_mask]
X_val_reduced = X_val_processed[:, kept_features_mask]
X_test_reduced = X_test_processed[:, kept_features_mask]

reduced_model = LogisticRegression(
    max_iter=500, solver="lbfgs", class_weight="balanced", random_state=42
)
reduced_model.fit(X_train_reduced, y_train)

metrics = evaluate_classification(reduced_model, X_val_reduced, y_val, binary=False)

train_loss_reduced = log_loss(
    y_train, reduced_model.predict_proba(X_train_reduced), labels=reduced_model.classes_
)
val_loss_reduced = log_loss(
    y_val, reduced_model.predict_proba(X_val_reduced), labels=reduced_model.classes_
)

print(
    f"Reduced-feature model | Train loss: {train_loss_reduced:.4f} | "
    f"Val loss: {val_loss_reduced:.4f}"
)
print(f"Features used: {X_train_reduced.shape[1]} (down from {X_train_processed.shape[1]})")
print(metrics)
print(classification_report(y_val, reduced_model.predict(X_val_reduced)))

## Multiclass Logistic Regression with PCA Features

In [ ]:
pca_df = pd.read_csv("output/results_pca/pca_component_scores.csv")
pca_df.columns

In [ ]:
pca_df.split.value_counts()

In [ ]:
pca_reg = pca_df.copy()
pca_reg = pca_reg.drop(columns=["approved_flag", "approved_flag_binary", "credit_score"])
y_pca_train = pca_df[pca_df.split == "train"]["approved_flag"]
y_pca_val = pca_df[pca_df.split == "val"]["approved_flag"]
y_pca_test = pca_df[pca_df.split == "test"]["approved_flag"]

In [ ]:
pca_train = pca_reg[pca_reg.split == "train"]
pca_val = pca_reg[pca_reg.split == "val"]
pca_test = pca_reg[pca_reg.split == "test"]
pca_train = pca_train.drop(columns=["split"])
pca_val = pca_val.drop(columns=["split"])
pca_test = pca_test.drop(columns=["split"])

In [ ]:
pca_model = LogisticRegression(
    max_iter=500, solver="lbfgs", class_weight="balanced", random_state=42
)
pca_model.fit(pca_train, y_pca_train)
y_pca_pred = pca_model.predict(pca_val)
y_pca_prob = pca_model.predict_proba(pca_val)

In [ ]:
evaluate_classification(pca_model, pca_test, y_pca_test, binary=False)
print(classification_report(y_pca_test, pca_model.predict(pca_test)))

## Below Precision-Recall plot shows whether P3 performance holds up after reducing to the PCA feature set (balanced class weights)

In [ ]:
pca_classes = pca_model.classes_
y_pca_val_bin = label_binarize(y_pca_val, classes=pca_classes)

fig, axes = plt.subplots(1, len(pca_classes), figsize=(5 * len(pca_classes), 5), sharey=True)
for ax, cls_idx, cls in zip(axes, range(len(pca_classes)), pca_classes):
    PrecisionRecallDisplay.from_predictions(
        y_pca_val_bin[:, cls_idx], y_pca_prob[:, cls_idx], ax=ax, name="PCA model"
    )
    ax.set_title(f"Class {cls} (one-vs-rest)")
plt.tight_layout()
plt.show()

## Summary: Effect of `class_weight` on P3 (before vs after)

- **Before (`class_weight=None`)**: the loss is dominated by the majority class (typically P2), so the model has little pressure to separate P3 from its neighbours. Expect P3's **recall** to be the weakest of the four classes in the baseline confusion matrix / classification report above, with many P3 rows misclassified into the adjacent P1 or P4 buckets.
- **After (`class_weight="balanced"`)**: each class contributes to the loss in inverse proportion to its frequency, so P3 (and any other minority class) is up-weighted during training. This typically **raises P3's recall** (fewer P3 customers slip through misclassified), at some cost to **precision** for P3 and to the majority class's own metrics, since the decision boundary shifts to accommodate the minority classes.
- Which trade-off is preferable is a business decision: if misclassifying a genuinely risky/marginal (P3) customer as low-risk is costlier than the reverse, `class_weight="balanced"` (or custom weights that penalise P3 errors even more heavily) is usually worth the small overall-accuracy trade-off. The grid search above lets the data decide by searching over both settings and scoring on macro-F1, which treats every class — including P3 — equally.

Artificially remove 10% of values from one column. Compare median vs KNN imputation. Which gives lower mean absolute error against the true values?
●
Detect outliers in one numerical column using IQR. Apply winsorisation. Plot the distribution before and after.
Task
●
Write a data quality report: which columns have missing values, what percentage, the imputation strategy chosen for each, and why.
● Impact of using winsorisation. When to use it. What would be better alternatives? Hint: Alternatives: medians, M-estimators, or quantile regression